# 🎧 Simple Content-Based Multi-Modal Music Recommender
### Blending Audio Acoustics & Lyric Themes to Recommend Similar Songs

This notebook builds a simple and practical **content-based recommendation engine** that combines:
1. **Audio Similarity** (using LAION-CLAP zero-shot acoustic representations)
2. **Lyric Similarity** (using Multilingual-E5-Large semantic representations)
3. **Smart Duplicate Filtering** (removing near-identical remixes/versions of the same artist)


## 1. Setup & Data Loading

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Robust path resolution
if Path('data/metadata/songs.parquet').exists():
    DATA_DIR = Path('data')
elif Path('../data/metadata/songs.parquet').exists():
    DATA_DIR = Path('../data')
elif Path('/kaggle/input/spotify-10k-music-features').exists():
    DATA_DIR = Path('/kaggle/input/spotify-10k-music-features')
else:
    raise FileNotFoundError("Dataset path not found.")

songs = pd.read_parquet(DATA_DIR / 'metadata' / 'songs.parquet')
dsp = pd.read_parquet(DATA_DIR / 'features' / 'audio' / 'dsp_librosa.parquet')
emotions = pd.read_parquet(DATA_DIR / 'features' / 'lyric' / 'go_emotions.parquet')

# Load normalized embeddings
clap = np.load(DATA_DIR / 'embeddings' / 'audio' / 'clap_512d.npy')
lyrics_e5 = np.load(DATA_DIR / 'embeddings' / 'lyric' / 'multilingual_e5_large_1024d.npy')

# L2 Normalize vectors for fast cosine similarity via dot product
clap_norm = clap / np.maximum(np.linalg.norm(clap, axis=1, keepdims=True), 1e-8)
lyrics_norm = lyrics_e5 / np.maximum(np.linalg.norm(lyrics_e5, axis=1, keepdims=True), 1e-8)

print(f"Loaded metadata and normalized embeddings for {len(songs):,} songs.")


Loaded metadata and normalized embeddings for 10,000 songs.


## 2. Multi-Modal Recommendation Function
We compute cosine similarity across both audio and lyrics, allowing a configurable weighting parameter:
$$\text{Score} = \alpha \cdot \text{Sim}_{\text{audio}} + (1 - \alpha) \cdot \text{Sim}_{\text{lyric}}$$

We also apply a simple filter to exclude the seed song itself and redundant versions/remixes from the same primary artist.


In [2]:
def recommend_songs(seed_idx: int, top_k: int = 5, audio_weight: float = 0.6):
    """
    Recommends top_k songs based on combined audio and lyric cosine similarity.
    """
    seed_song = songs.iloc[seed_idx]
    seed_artist = str(seed_song['artist_names']).lower().split(',')[0].strip()
    seed_title = str(seed_song['track_name']).lower()
    
    # 1. Compute cosine similarities across full 10k corpus
    sim_audio = np.dot(clap_norm, clap_norm[seed_idx])
    sim_lyric = np.dot(lyrics_norm, lyrics_norm[seed_idx])
    
    # 2. Blend scores
    combined_scores = (audio_weight * sim_audio) + ((1.0 - audio_weight) * sim_lyric)
    
    # 3. Sort indices
    ranked_indices = np.argsort(combined_scores)[::-1]
    
    # 4. Filter duplicates (same title or same primary artist with variant title)
    recommendations = []
    for idx in ranked_indices:
        if idx == seed_idx:
            continue
        
        cand_song = songs.iloc[idx]
        cand_artist = str(cand_song['artist_names']).lower().split(',')[0].strip()
        cand_title = str(cand_song['track_name']).lower()
        
        # Simple duplicate check
        if cand_artist == seed_artist and (seed_title in cand_title or cand_title in seed_title):
            continue
            
        recommendations.append({
            'row_idx': idx,
            'track_name': cand_song['track_name'],
            'artist': cand_song['artist_names'],
            'genre': cand_song['main_genres'],
            'audio_sim': round(float(sim_audio[idx]), 3),
            'lyric_sim': round(float(sim_lyric[idx]), 3),
            'combined_score': round(float(combined_scores[idx]), 3)
        })
        
        if len(recommendations) >= top_k:
            break
            
    return pd.DataFrame(recommendations)


## 3. Testing Recommendations on Sample Tracks

In [3]:
# Example 1: Seed Track 15
seed_idx = 15
seed_info = songs.iloc[seed_idx]
print(f"🎵 SEED TRACK: '{seed_info['track_name']}' by {seed_info['artist_names']} (Genre: {seed_info['main_genres']})")
print("-" * 80)

# Get top 5 recommendations (60% audio vibe, 40% lyric theme)
recs = recommend_songs(seed_idx=seed_idx, top_k=5, audio_weight=0.6)
recs[['track_name', 'artist', 'genre', 'audio_sim', 'lyric_sim', 'combined_score']]


🎵 SEED TRACK: 'Good Luck, Babe!' by Chappell Roan (Genre: Pop, Rock)
--------------------------------------------------------------------------------


,track_name,artist,genre,audio_sim,lyric_sim,combined_score
0,Busy Woman,Sabrina Carpenter,Pop,0.920,0.880,0.904
1,Recommence-moi,SANTA,"Folk, Pop, Traditional Music",0.929,0.861,0.902
2,Leave The Door Open,Bruno Mars|Anderson .Paak|Silk Sonic,"Electronic, Hip Hop, Pop, R&B",0.886,0.890,0.887
3,Fernando,ABBA,Pop,0.905,0.856,0.885
4,I Try,Macy Gray,R&B,0.887,0.883,0.885


## 4. Comparing Audio-Focused vs Lyric-Focused Recommendations
Adjusting the `audio_weight` allows users to prioritize musical acoustic similarity versus lyrical storytelling.


In [4]:
# print seed again
print(f"\n🎵 SEED TRACK: '{seed_info['track_name']}' by {seed_info['artist_names']} (Genre: {seed_info['main_genres']})")
print()

# Audio-focused (85% audio, 15% lyric)
print("🔊 AUDIO-FOCUSED (Acoustic Match):")
recs_audio = recommend_songs(seed_idx=seed_idx, top_k=5, audio_weight=0.85)  # audio 85%
display(recs_audio[['track_name', 'artist', 'genre', 'audio_sim', 'lyric_sim', 'combined_score']])

# Lyric-focused (15% audio, 85% lyric)
print("\n📖 LYRIC-FOCUSED (Thematic Match):")
recs_lyric = recommend_songs(seed_idx=seed_idx, top_k=5, audio_weight=0.15)  # lyric 85%
display(recs_lyric[['track_name', 'artist', 'genre', 'audio_sim', 'lyric_sim', 'combined_score']])



🎵 SEED TRACK: 'Good Luck, Babe!' by Chappell Roan (Genre: Pop, Rock)

🔊 AUDIO-FOCUSED (Acoustic Match):


,track_name,artist,genre,audio_sim,lyric_sim,combined_score
0,Recommence-moi,SANTA,"Folk, Pop, Traditional Music",0.929,0.861,0.919
1,Busy Woman,Sabrina Carpenter,Pop,0.920,0.880,0.914
2,Fernando,ABBA,Pop,0.905,0.856,0.898
3,Tu Dama De Hierro,Marisela,"Folk, Latin, Pop",0.895,0.858,0.890
4,Cuando Baja La Marea (feat. Regina Blandón),Mentiras: La Serie|Belinda|Mariana Treviño|Dia...,"Folk, Latin, Pop, Traditional Music",0.900,0.825,0.889



📖 LYRIC-FOCUSED (Thematic Match):


,track_name,artist,genre,audio_sim,lyric_sim,combined_score
0,Lady Of Namek,Tory Lanez,"Hip Hop, Pop, R&B",0.869,0.896,0.892
1,Leave The Door Open,Bruno Mars|Anderson .Paak|Silk Sonic,"Electronic, Hip Hop, Pop, R&B",0.886,0.890,0.889
2,Please Please Please,Sabrina Carpenter,Pop,0.862,0.893,0.888
3,Too Good to Say Goodbye,Bruno Mars,"Electronic, Pop",0.812,0.900,0.887
4,Busy Woman,Sabrina Carpenter,Pop,0.920,0.880,0.886


## 5. Mini Playlist Sequence Generator
Building a smooth sequential playlist where each track transitions seamlessly into the next.


In [5]:
def generate_playlist(start_seed_idx: int, playlist_length: int = 5):
    playlist = [start_seed_idx]
    visited = {start_seed_idx}
    
    current_idx = start_seed_idx
    for step in range(playlist_length - 1):
        candidates = recommend_songs(seed_idx=current_idx, top_k=15, audio_weight=0.7)
        # Pick top candidate not already in playlist
        next_idx = None
        for _, row in candidates.iterrows():
            cand_id = int(row['row_idx'])
            if cand_id not in visited:
                next_idx = cand_id
                break
        if next_idx is None:
            break
        playlist.append(next_idx)
        visited.add(next_idx)
        current_idx = next_idx
        
    playlist_df = songs.iloc[playlist][['track_id', 'track_name', 'artist_names', 'main_genres', 'release_date']].copy().reset_index(drop=True)
    playlist_df.index = [f"Track {i+1}" for i in range(len(playlist_df))]
    return playlist_df

print("🎶 Generated Smooth Mini-Playlist:")

seed_idx = 25
# seed song:
# also add spotify id to table
print(f"🎵 SEED TRACK: '{songs.iloc[seed_idx]['track_id']}' {songs.iloc[seed_idx]['track_name']}' by {songs.iloc[seed_idx]['artist_names']} (Genre: {songs.iloc[seed_idx]['main_genres']})")

generate_playlist(start_seed_idx=seed_idx, playlist_length=5)

# How is this work: 
# get track 


🎶 Generated Smooth Mini-Playlist:
🎵 SEED TRACK: '5XeFesFbtLpXzIVDNQP22n' I Wanna Be Yours' by Arctic Monkeys (Genre: Pop, Rock)


,track_id,track_name,artist_names,main_genres,release_date
Track 1,5XeFesFbtLpXzIVDNQP22n,I Wanna Be Yours,Arctic Monkeys,"Pop, Rock",2013-09-09
Track 2,2xql0pid3EUwW38AsywxhV,Reflections,The Neighbourhood,"Pop, Rock",2018-11-02
Track 3,113xf7t4qNM7038YJvauik,Nervous,The Neighbourhood,"Pop, Rock",2018-11-02
Track 4,5Q6fh8OEhBYepJaORz9lxe,Daddy Issues (Remix) feat. Syd,The Neighbourhood|Syd,"Pop, R&B, Rock",2015-11-13
Track 5,5E30LdtzQTGqRvNd7l6kG5,Daddy Issues,The Neighbourhood,"Pop, Rock",2015-10-30
